In [3]:
# ────────────────────────────────────────────────────────────────────────────────
# Example usage
# ────────────────────────────────────────────────────────────────────────────────
# >> preview_slideshow("my_dataset", delay=1.5, recursive=True)
# >> print("Min size:", min_image_size("my_dataset", recursive=True))

In [5]:
print("Min size:", min_image_size("/Users/davidschneider/data/image/comics/ao", recursive=True))

Min size: (734, 242)


In [6]:
preview_slideshow("/Users/davidschneider/data/image/comics/ao", delay=0.1, recursive=True)

Done! Displayed 73 images.


# Train VAE on 240 x 240 patches

In [7]:
#!/usr/bin/env python
# -*- coding: utf-8 -*-

import math, random, argparse, itertools
from pathlib import Path

import torch
import torch.nn as nn
import torchvision.transforms as T
from torch.utils.data import Dataset, DataLoader, random_split
from torch.utils.tensorboard import SummaryWriter
from PIL import Image

# ───────────────────────────────────────────────────────────────────────────────
# 0. Hyper‑parameters (you can override from CLI)
# ───────────────────────────────────────────────────────────────────────────────
H = dict(
    image_dir="/Users/davidschneider/data/image/comics/ao",   # folder with .jpg / .gif …
    epochs=200,
    lr=2e-4,
    batch_size=64,                # 64 crops per step; GPU‑memory friendly
    latent_dim=128,               # size of the z vector
    beta=1.0,                     # KL weight (β‑VAE)
    num_workers=4,
    logdir="runs/vae240",
    checkpoint="vae_checkpoint.pt",
    crop_size=240,
    val_split=0.2,
    seed=42,
)

# ───────────────────────────────────────────────────────────────────────────────
# 1. Dataset
# ───────────────────────────────────────────────────────────────────────────────
class CroppedImageDataset(Dataset):
    def __init__(self, files, crop_size=240):
        self.files = files
        self.crop_size = crop_size
        self.transform = T.Compose([
            T.RandomCrop(crop_size),          # data augmentation
            T.ToTensor(),                     # (0,1)
        ])

    def __len__(self):
        return len(self.files)

    def __getitem__(self, idx):
        path = self.files[idx]
        img = Image.open(path).convert("RGB")
        return self.transform(img)

def make_datasets(image_root: str | Path, val_split: float, crop_size: int):
    img_dir = Path(image_root).expanduser()
    files = sorted([p for p in img_dir.iterdir() if p.suffix.lower() in
                    {".jpg", ".jpeg", ".gif", ".png"}])
    assert files, f"No images found in {img_dir}"
    random.Random(H["seed"]).shuffle(files)

    n_val = math.ceil(len(files) * val_split)
    val_files, train_files = files[:n_val], files[n_val:]
    train_ds = CroppedImageDataset(train_files, crop_size)
    val_ds   = CroppedImageDataset(val_files, crop_size)
    return train_ds, val_ds

# ───────────────────────────────────────────────────────────────────────────────
# 2. Very small Conv VAE (downsample ×8 overall ⇒ latent 30×30)
# ───────────────────────────────────────────────────────────────────────────────
class Encoder(nn.Module):
    def __init__(self, latent_dim):
        super().__init__()
        self.conv = nn.Sequential(
            nn.Conv2d(3,   32, 4, 2, 1), nn.ReLU(),   # 240 → 120
            nn.Conv2d(32,  64, 4, 2, 1), nn.ReLU(),   # 120 → 60
            nn.Conv2d(64, 128, 4, 2, 1), nn.ReLU(),   # 60  → 30
        )
        self.mu_head  = nn.Conv2d(128, latent_dim, 1)
        self.logvar_head = nn.Conv2d(128, latent_dim, 1)

    def forward(self, x):
        h = self.conv(x)
        return self.mu_head(h), self.logvar_head(h)     # N × C × 30 × 30


class Decoder(nn.Module):
    def __init__(self, latent_dim):
        super().__init__()
        self.deconv = nn.Sequential(
            nn.ConvTranspose2d(latent_dim, 128, 4, 2, 1), nn.ReLU(),  # 30→60
            nn.ConvTranspose2d(128, 64, 4, 2, 1), nn.ReLU(),          # 60→120
            nn.ConvTranspose2d(64, 32, 4, 2, 1), nn.ReLU(),           # 120→240
            nn.Conv2d(32, 3, 3, 1, 1), nn.Sigmoid(),                  # (0,1)
        )

    def forward(self, z):
        return self.deconv(z)


class VAE(nn.Module):
    def __init__(self, latent_dim):
        super().__init__()
        self.enc = Encoder(latent_dim)
        self.dec = Decoder(latent_dim)

    def reparameterize(self, mu, logvar):
        std = torch.exp(0.5 * logvar)
        eps = torch.randn_like(std)
        return mu + eps * std

    def forward(self, x):
        mu, logvar = self.enc(x)
        z = self.reparameterize(mu, logvar)
        return self.dec(z), mu, logvar

# ───────────────────────────────────────────────────────────────────────────────
# 3. Training utilities
# ───────────────────────────────────────────────────────────────────────────────
def loss_fn(x, x_hat, mu, logvar, beta=1.0):
    recon = torch.nn.functional.l1_loss(x_hat, x, reduction="mean")
    kl = -0.5 * torch.mean(1 + logvar - mu.pow(2) - logvar.exp())
    return recon + beta * kl, recon, kl

def run_epoch(
    model, loader, opt, device, epoch, writer, split: str, beta: float, log_every=100
):
    is_train = split == "train"
    model.train(is_train)
    running = {"loss": 0.0, "recon": 0.0, "kl": 0.0}

    for step, batch in enumerate(loader):
        x = batch.to(device)
        x_hat, mu, logvar = model(x)
        loss, recon, kl = loss_fn(x, x_hat, mu, logvar, beta)

        if is_train:
            opt.zero_grad()
            loss.backward()
            opt.step()

        # log
        for k, v in [("loss", loss), ("recon", recon), ("kl", kl)]:
            running[k] += v.item() * x.size(0)

        global_step = epoch * len(loader) + step
        if is_train and step % log_every == 0:
            writer.add_scalar(f"{split}/loss", loss, global_step)

    n = len(loader.dataset)
    epoch_metrics = {k: v / n for k, v in running.items()}
    for k, v in epoch_metrics.items():
        writer.add_scalar(f"{split}/{k}", v, epoch)
    return epoch_metrics["loss"]

# ───────────────────────────────────────────────────────────────────────────────
# 4. Main
# ───────────────────────────────────────────────────────────────────────────────
def main(cfg=H):
    random.seed(cfg["seed"])
    torch.manual_seed(cfg["seed"])

    train_ds, val_ds = make_datasets(
        cfg["image_dir"], cfg["val_split"], cfg["crop_size"]
    )
    train_dl = DataLoader(
        train_ds, cfg["batch_size"], shuffle=True,
        num_workers=cfg["num_workers"], pin_memory=True
    )
    val_dl = DataLoader(
        val_ds, cfg["batch_size"], shuffle=False,
        num_workers=cfg["num_workers"], pin_memory=True
    )

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model = VAE(cfg["latent_dim"]).to(device)
    opt = torch.optim.AdamW(model.parameters(), lr=cfg["lr"])
    writer = SummaryWriter(cfg["logdir"])

    best_val = float("inf")
    for epoch in range(cfg["epochs"]):
        train_loss = run_epoch(
            model, train_dl, opt, device, epoch, writer, "train", cfg["beta"]
        )
        val_loss   = run_epoch(
            model, val_dl,  None, device, epoch, writer, "val",   cfg["beta"]
        )

        print(f"Epoch {epoch:03d}  train {train_loss:.4f}  val {val_loss:.4f}")

        if val_loss < best_val:
            best_val = val_loss
            torch.save(model.state_dict(), cfg["checkpoint"])
            print(f"  ✔ Saved checkpoint ({best_val:.4f})")

    writer.close()
    print("Training complete. Launch TensorBoard with:\n"
          f"    tensorboard --logdir {cfg['logdir']}")


In [11]:
cfg=H
random.seed(cfg["seed"])
torch.manual_seed(cfg["seed"])

In [12]:
    train_ds, val_ds = make_datasets(
        cfg["image_dir"], cfg["val_split"], cfg["crop_size"]
    )
    train_dl = DataLoader(
        train_ds, cfg["batch_size"], shuffle=True,
        num_workers=cfg["num_workers"], pin_memory=True
    )
    val_dl = DataLoader(
        val_ds, cfg["batch_size"], shuffle=False,
        num_workers=cfg["num_workers"], pin_memory=True
    )


In [15]:
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model = VAE(cfg["latent_dim"]).to(device)
    opt = torch.optim.AdamW(model.parameters(), lr=cfg["lr"])
    writer = SummaryWriter(cfg["logdir"])

    best_val = float("inf")


In [14]:
cfg['num_workers']

4

In [16]:
epoch = 1

In [17]:
        train_loss = run_epoch(
            model, train_dl, opt, device, epoch, writer, "train", cfg["beta"]
        )


Traceback (most recent call last):
  File "<string>", line 1, in <module>
  File "/Users/davidschneider/.local/share/uv/python/cpython-3.12.8-macos-aarch64-none/lib/python3.12/multiprocessing/spawn.py", line 122, in spawn_main
Traceback (most recent call last):
  File "<string>", line 1, in <module>
  File "/Users/davidschneider/.local/share/uv/python/cpython-3.12.8-macos-aarch64-none/lib/python3.12/multiprocessing/spawn.py", line 122, in spawn_main
    exitcode = _main(fd, parent_sentinel)
    exitcode = _main(fd, parent_sentinel)
               ^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/Users/davidschneider/.local/share/uv/python/cpython-3.12.8-macos-aarch64-none/lib/python3.12/multiprocessing/spawn.py", line 132, in _main
               ^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/Users/davidschneider/.local/share/uv/python/cpython-3.12.8-macos-aarch64-none/lib/python3.12/multiprocessing/spawn.py", line 132, in _main
    self = reduction.pickle.load(from_parent)
           ^^^^^^^^^^^^^^^^^^^^^^^^^

RuntimeError: DataLoader worker (pid(s) 75297) exited unexpectedly

In [18]:
for x in train_dl:
    break


Traceback (most recent call last):
  File "<string>", line 1, in <module>
  File "/Users/davidschneider/.local/share/uv/python/cpython-3.12.8-macos-aarch64-none/lib/python3.12/multiprocessing/spawn.py", line 122, in spawn_main
    exitcode = _main(fd, parent_sentinel)
               ^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/Users/davidschneider/.local/share/uv/python/cpython-3.12.8-macos-aarch64-none/lib/python3.12/multiprocessing/spawn.py", line 132, in _main
    self = reduction.pickle.load(from_parent)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
AttributeError: Can't get attribute 'CroppedImageDataset' on <module '__main__' (<class '_frozen_importlib.BuiltinImporter'>)>
Traceback (most recent call last):
  File "<string>", line 1, in <module>
  File "/Users/davidschneider/.local/share/uv/python/cpython-3.12.8-macos-aarch64-none/lib/python3.12/multiprocessing/spawn.py", line 122, in spawn_main
    exitcode = _main(fd, parent_sentinel)
               ^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/Us

RuntimeError: DataLoader worker (pid(s) 75368) exited unexpectedly